# 📄 ArXiv AI/ML Papers — Research Category Classifier
### Predict which AI field a paper belongs to using its abstract text
---
**What you'll learn in this notebook:**
- How to explore a real-world NLP dataset
- How to turn raw text into numbers using TF-IDF
- How to train a machine learning model to classify research papers
- How to evaluate your model properly

> 🎯 **Goal:** Given the abstract of an AI paper, predict its primary arXiv category (cs.CV, cs.LG, cs.CL, cs.RO, or cs.AI)


---
## 📦 Step 1 — Import Libraries

We start by loading all the tools we'll need.  
Think of libraries as toolboxes — each one gives us ready-made functions.


In [ ]:
# 📚 Data handling
import pandas as pd
import numpy as np

# 📊 Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# 🤖 Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

# ⚙️ Utilities
import warnings
warnings.filterwarnings("ignore")

# 🎨 Plot style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print("✅ All libraries imported successfully!")


---
## 📂 Step 2 — Load the Dataset

We load the CSV file into a **DataFrame** — a table with rows and columns, like an Excel sheet in Python.


In [ ]:
df = pd.read_csv("/kaggle/input/arxiv-aiml-papers-2025-2026/arxiv_ai_ml_papers_cleaned.csv")

print(f"✅ Dataset loaded!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")


In [ ]:
# Preview the first 3 rows
df.head(3)


---
## 🔍 Step 3 — Exploratory Data Analysis (EDA)

Before building any model, we need to **understand our data**.  
EDA helps us answer: *What does the data look like? Are there any patterns?*


### 3.1 — Dataset Overview

In [ ]:
# Check data types and non-null counts
df.info()


In [ ]:
# Statistical summary of numeric columns
df[["num_authors","abstract_length","word_count",
    "num_categories","update_lag_days","days_since_submission"]].describe().round(2)


### 3.2 — Missing Values Check
Our dataset was pre-cleaned, so we expect zero real missing values.


In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "✅ No missing values found — dataset is clean!")


### 3.3 — Target Column: Research Category Distribution
Our **target** is `primary_category` — what we want to predict.  
Let's see how papers are distributed across categories.


In [ ]:
cat_counts = df["primary_category"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — top 10 categories
top10 = cat_counts.head(10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color=sns.color_palette("Set2", 10))
axes[0].set_xlabel("Number of Papers")
axes[0].set_title("Top 10 Research Categories")
for i, v in enumerate(top10.values[::-1]):
    axes[0].text(v + 10, i, str(v), va="center", fontsize=9)

# Pie chart — top 5 categories
top5_counts = cat_counts.head(5)
axes[1].pie(top5_counts.values, labels=top5_counts.index,
            autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("Set2", 5))
axes[1].set_title("Top 5 Categories — Share of Papers")

plt.tight_layout()
plt.savefig("category_distribution.png", bbox_inches="tight")
plt.show()
print(f"\n📊 Total unique categories: {df['primary_category'].nunique()}")


### 3.4 — Papers Over Time
Let's see how many papers were submitted each month.


In [ ]:
df["submitted_date"] = pd.to_datetime(df["submitted_date"])
monthly = df.groupby(df["submitted_date"].dt.to_period("M")).size().reset_index()
monthly.columns = ["month", "count"]
monthly["month_str"] = monthly["month"].astype(str)

plt.figure(figsize=(12, 4))
plt.bar(monthly["month_str"], monthly["count"],
        color=sns.color_palette("Set2", len(monthly)))
plt.xlabel("Month")
plt.ylabel("Papers Submitted")
plt.title("Monthly Paper Submissions — arXiv AI/ML (2025–2026)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("monthly_submissions.png", bbox_inches="tight")
plt.show()


### 3.5 — Abstract Length Distribution
Longer abstracts sometimes give more clues about the research field.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(df["word_count"], bins=40, color="#4CAF91", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Number of Papers")
axes[0].set_title("Distribution of Abstract Word Count")
axes[0].axvline(df["word_count"].mean(), color="red", linestyle="--", label=f"Mean: {df['word_count'].mean():.0f}")
axes[0].legend()

# Boxplot by top 5 category
top5_cats = df["primary_category"].value_counts().head(5).index
df_top5 = df[df["primary_category"].isin(top5_cats)]
df_top5.boxplot(column="word_count", by="primary_category",
                ax=axes[1], patch_artist=True)
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Word Count")
axes[1].set_title("Abstract Length by Category")
plt.suptitle("")  # remove default title
plt.tight_layout()
plt.savefig("abstract_length.png", bbox_inches="tight")
plt.show()

print(f"Average word count : {df['word_count'].mean():.0f} words")
print(f"Min word count     : {df['word_count'].min()} words")
print(f"Max word count     : {df['word_count'].max()} words")


### 3.6 — Author Collaboration Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram — capped at 20 authors for readability
capped = df["num_authors"].clip(upper=20)
axes[0].hist(capped, bins=20, color="#5B8DB8", edgecolor="white")
axes[0].set_xlabel("Number of Authors (capped at 20)")
axes[0].set_ylabel("Papers")
axes[0].set_title("Author Count Distribution")
axes[0].axvline(df["num_authors"].median(), color="red", linestyle="--",
                label=f"Median: {df['num_authors'].median():.0f}")
axes[0].legend()

# is_updated flag
updated_counts = df["is_updated"].value_counts()
labels = ["Not Revised (0)", "Revised (1)"]
axes[1].bar(labels, updated_counts.values,
            color=["#4CAF91", "#E07B54"])
axes[1].set_ylabel("Papers")
axes[1].set_title("Papers Revised After Submission")
for i, v in enumerate(updated_counts.values):
    axes[1].text(i, v + 30, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("collaboration.png", bbox_inches="tight")
plt.show()


---
## 🛠️ Step 4 — Prepare Data for Modelling

We'll focus on the **top 5 categories** to keep the model clean and accurate.  
Our features = abstract text | Our target = primary category


In [ ]:
# ── Keep only top 5 categories ──────────────────────────────
TOP5 = df["primary_category"].value_counts().head(5).index.tolist()
print(f"Top 5 categories: {TOP5}")

df_model = df[df["primary_category"].isin(TOP5)].copy()
print(f"\nRows kept for modelling: {len(df_model):,}")
print(df_model["primary_category"].value_counts())


In [ ]:
# ── Feature & Target ────────────────────────────────────────
X = df_model["abstract"]          # raw text — what we feed to the model
y = df_model["primary_category"]  # label   — what we want to predict

print(f"Feature (X): abstract text — {len(X):,} rows")
print(f"Target  (y): {y.nunique()} unique classes → {y.unique().tolist()}")


---
## ✂️ Step 5 — Train / Test Split

We split the data into:
- **Training set (80%)** — what the model learns from
- **Test set (20%)** — what we use to check how well the model learned

> 💡 We use `stratify=y` so each category has the same ratio in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42,    # fixed seed for reproducibility
    stratify=y          # keep class proportions balanced
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing  samples : {len(X_test):,}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())


---
## 🔢 Step 6 — TF-IDF Vectorisation

Computers can't read text — they only understand numbers.  
**TF-IDF (Term Frequency–Inverse Document Frequency)** converts each abstract into a numeric vector.

| Term | Meaning |
|------|---------|
| **TF** | How often a word appears in *this* document |
| **IDF** | How rare the word is *across all* documents |
| **TF-IDF** | High score = word is important here but rare elsewhere |

> Example: the word *"robot"* appearing in a robotics paper gets a high score because it's rare in NLP papers.


In [ ]:
# TF-IDF converts text → numbers the model can learn from
# max_features=10000 → keep top 10,000 most informative words/phrases
# ngram_range=(1,2)  → use single words AND 2-word phrases ("neural network")
# sublinear_tf=True  → dampen very frequent words (log scale)

vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),     # unigrams + bigrams
    sublinear_tf=True,      # log normalisation
    min_df=2                # ignore words appearing in < 2 docs
)

X_train_vec = vectorizer.fit_transform(X_train)   # learn vocabulary + transform
X_test_vec  = vectorizer.transform(X_test)         # transform only (no re-learning)

print(f"Training matrix shape : {X_train_vec.shape}")
print(f"  → {X_train_vec.shape[0]:,} papers × {X_train_vec.shape[1]:,} features")
print(f"\nSample top features: {vectorizer.get_feature_names_out()[:10].tolist()}")


---
## 🤖 Step 7 — Train the Model

We use **Logistic Regression** — a simple but powerful algorithm for text classification.  
It works by learning which words/phrases are most associated with each category.

> **Why Logistic Regression for text?**  
> It's fast, interpretable, and works extremely well with TF-IDF features.


In [ ]:
model = LogisticRegression(
    C=5,             # regularisation strength (higher = less penalty)
    max_iter=1000,   # maximum training iterations
    random_state=42,
    solver="lbfgs",
    multi_class="auto"
)

model.fit(X_train_vec, y_train)
print("✅ Model trained successfully!")

# Quick check on training set
train_acc = accuracy_score(y_train, model.predict(X_train_vec))
print(f"   Training Accuracy : {train_acc:.4f} ({train_acc*100:.2f}%)")


---
## 📊 Step 8 — Evaluate the Model

Now we test the model on data it has **never seen before** (the test set).


In [ ]:
y_pred = model.predict(X_test_vec)
test_acc = accuracy_score(y_test, y_pred)

print("=" * 50)
print(f"  ✅ Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print("=" * 50)


### 8.1 — Classification Report

The report shows 3 key metrics per class:

| Metric | Meaning |
|--------|---------|
| **Precision** | Of all papers predicted as X, how many actually are X? |
| **Recall** | Of all real X papers, how many did we correctly find? |
| **F1-Score** | Harmonic mean of Precision & Recall (the balanced score) |


In [ ]:
print("\nClassification Report:")
print("-" * 60)
print(classification_report(y_test, y_pred, digits=4))


### 8.2 — Confusion Matrix

A confusion matrix shows where the model makes mistakes.  
**Diagonal = correct predictions** | **Off-diagonal = errors**


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Confusion Matrix — Category Classifier", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("confusion_matrix.png", bbox_inches="tight")
plt.show()
print("\n💡 Darker diagonal = better predictions")


### 8.3 — Top Predictive Words per Category

Let's look at which words the model relies on most for each category.


In [ ]:
feature_names = vectorizer.get_feature_names_out()
n_top = 12

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, cls in zip(axes, model.classes_):
    idx = list(model.classes_).index(cls)
    coef = model.coef_[idx]
    top_idx = coef.argsort()[-n_top:][::-1]
    top_words = feature_names[top_idx]
    top_scores = coef[top_idx]

    ax.barh(top_words[::-1], top_scores[::-1],
            color=sns.color_palette("Set2", n_top))
    ax.set_title(cls, fontsize=11, fontweight="bold")
    ax.set_xlabel("TF-IDF Weight")
    ax.tick_params(labelsize=8)

plt.suptitle("Top 12 Predictive Words per Category", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("top_words.png", bbox_inches="tight")
plt.show()


---
## 🔮 Step 9 — Make Predictions on New Abstracts

Let's use our trained model to classify brand-new abstracts!


In [ ]:
def predict_category(abstract_text):
    """Predict the arXiv category for a given abstract."""
    vec = vectorizer.transform([abstract_text])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    conf = max(proba) * 100

    print(f"Predicted Category : {pred}")
    print(f"Confidence         : {conf:.1f}%")
    print("\nAll category probabilities:")
    for cat, p in sorted(zip(model.classes_, proba), key=lambda x: -x[1]):
        bar = "█" * int(p * 30)
        print(f"  {cat:10s} {bar:30s} {p*100:.1f}%")
    return pred

# ── Test 1: A Computer Vision abstract ──────────────────────
print("=" * 60)
print("TEST 1 — Computer Vision Paper")
print("=" * 60)
predict_category(
    "We propose a novel convolutional neural network architecture for "
    "real-time object detection and image segmentation. Our method "
    "achieves state-of-the-art performance on COCO benchmark with a "
    "mAP of 58.3, outperforming previous approaches by 3.2 points while "
    "running at 45 FPS on a single GPU."
)


In [ ]:
# ── Test 2: A Robotics abstract ─────────────────────────────
print("=" * 60)
print("TEST 2 — Robotics Paper")
print("=" * 60)
predict_category(
    "We present a reinforcement learning framework for autonomous robot "
    "navigation in unknown environments. The robot learns to plan collision-free "
    "paths using a combination of LiDAR sensor inputs and depth cameras. "
    "Our approach achieves 94% success rate in real-world maze environments."
)


In [ ]:
# ── Test 3: An NLP abstract ─────────────────────────────────
print("=" * 60)
print("TEST 3 — NLP / Language Model Paper")
print("=" * 60)
predict_category(
    "We introduce a new pre-training objective for large language models "
    "that improves performance on question answering and text summarisation tasks. "
    "Our method fine-tunes transformer-based architectures using contrastive "
    "sentence-level representations and achieves a ROUGE-L score of 47.8 on CNN/DailyMail."
)


---
## 📈 Step 10 — Model Performance Summary


In [ ]:
# Build a clean summary table
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=model.classes_
)

summary = pd.DataFrame({
    "Category":   model.classes_,
    "Precision":  (precision * 100).round(2),
    "Recall":     (recall * 100).round(2),
    "F1-Score":   (f1 * 100).round(2),
    "Support":    support.astype(int)
})

print("=" * 58)
print(f"  Overall Accuracy : {test_acc*100:.2f}%")
print("=" * 58)
print(summary.to_string(index=False))
print("-" * 58)
print(f"  Macro F1 Average : {f1.mean()*100:.2f}%")

# Visual bar chart
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(model.classes_))
w = 0.25
ax.bar([i - w for i in x], precision*100, width=w, label="Precision", color="#4CAF91")
ax.bar([i      for i in x], recall*100,    width=w, label="Recall",    color="#5B8DB8")
ax.bar([i + w for i in x], f1*100,         width=w, label="F1-Score",  color="#E07B54")
ax.set_xticks(x)
ax.set_xticklabels(model.classes_)
ax.set_ylim(0, 105)
ax.set_ylabel("Score (%)")
ax.set_title("Precision / Recall / F1 per Category")
ax.legend()
ax.axhline(test_acc*100, color="grey", linestyle="--", alpha=0.6, label="Accuracy")
plt.tight_layout()
plt.savefig("performance_summary.png", bbox_inches="tight")
plt.show()


---
## 🏁 Conclusion

In this notebook, we built a complete **NLP text classification pipeline** from scratch — from raw paper abstracts to a working research category predictor.

---

### 📋 What We Did

| Step | Action |
|------|--------|
| 1 | Loaded 7,701 real arXiv AI/ML papers |
| 2 | Explored data — distributions, trends, author patterns |
| 3 | Focused on top 5 research categories |
| 4 | Converted abstract text → TF-IDF numeric matrix |
| 5 | Trained a Logistic Regression classifier |
| 6 | Evaluated with accuracy, F1, and confusion matrix |
| 7 | Predicted categories for brand-new abstracts |

---

### 🏆 Final Model Performance

| Metric | Score |
|--------|-------|
| **Test Accuracy** | **~77.3%** |
| **Best Category** | cs.RO (F1 ≈ 87%) |
| **Hardest Category** | cs.AI (F1 ≈ 47%) |
| **Macro F1** | ~74% |

---

### 💡 Key Insights from the Data

- **cs.CV and cs.LG** dominate arXiv submissions — Computer Vision and Machine Learning are the most active fields
- **cs.RO (Robotics)** is the easiest to classify — its vocabulary (robot, manipulation, navigation) is very distinct
- **cs.AI vs cs.LG** are often confused — both deal with learning algorithms, making separation harder
- **10.9% of papers** were revised after submission, with revision lags up to 197 days
- Only **3.2% of papers** have a journal reference, confirming most are fresh preprints

---

### 🚀 How to Improve Further

1. **Use Title + Abstract combined** as input text instead of abstract alone
2. **Try XGBoost or LightGBM** with additional numeric features (word_count, num_authors)
3. **Fine-tune a BERT/SciBERT model** — transformer embeddings capture semantic meaning far better than TF-IDF
4. **Add all_categories** as multi-label features for richer context
5. **Expand to all 107 categories** for a harder, more comprehensive classifier

---

> ✅ **If you found this notebook useful, please give it an upvote!**  
> Feel free to fork it and experiment with different models.
